# 020 — Corridor rule only beyond the penalty-area line

Изменение относительно 019: автоматическое владение при мяче вне коридора применяется, только если линия выбранной команды строго дальше 16,5 м от её собственных ворот, в сторону центра. Проверяем соответствующую сторону поля отдельно для каждой команды и тайма. На границе, внутри штрафной и при отсутствии линии прямое назначение отключается. Требование относится к линии команды, которой назначался бы мяч; обе линии одновременно не обязаны быть выдвинуты.

Отключённые кадры возвращаются в обычную обработку 007: близость игроков к мячу, уверенные серии и разрешение спорных промежутков. Они не удаляются и не получают принудительную метку другой команды. Очистка координат сохраняется как в 007/019. Флаги допуска коридора применяются также в резервном заполнении пропусков, чтобы оно не обходило новое правило.

D: неизменные веса 016, порог 0,5, длительности из 019. Стратегия H/A и параметры зафиксированы, нового подбора нет. Сохраняется отсутствие X: заполнение по наблюдениям того же тайма, D вне игровых интервалов. Имеются синтетические проверки обеих сторон/границы, проверка неизменности D и покадрового контракта.

Run All как в 019; исходные локальные данные и подготовленные признаки обязательны. Выходы только local_review/experiments/020_penalty_gated_corridor. Прогноз и видео не публикуются. Модель 016 обучена на публичном префиксе, последняя минута неоднократно исследована; это офлайн-диагностика, не независимый тест.


In [ ]:
from pathlib import Path
import json,sys,hashlib,pickle,time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import GAME,GAMES,half_video_blocks,asset_paths
from evaluation import possession_metrics as pm
EXPERIMENT='020_penalty_gated_corridor'
OUT=ROOT/'local_review/experiments'/EXPERIMENT;OUT.mkdir(parents=True,exist_ok=True)
N=199319;FPS=25;BLOCKS=half_video_blocks(GAME)
def progress(stage,**kw):(OUT/'progress.json').write_text(json.dumps(dict(stage=stage,**kw),indent=2))
# Rebuild probabilities from frozen 016 weights, without fitting or reading private labels.
# Восстанавливаем вероятности из неизменных весов 016, без обучения и закрытой разметки.
progress('016 inference')
parent016=json.loads((ROOT/'sol/016_dead_broadcast_features/experiment.ipynb').read_text())
denv={'EXPERIMENT':EXPERIMENT}
exec(''.join(next(c for c in parent016['cells'] if c.get('id')=='phase-0')['source']),denv)
with (ROOT/'sol/016_dead_broadcast_features/public_model.pkl').open('rb') as f:dmodel=pickle.load(f)
probability=denv['predict'](dmodel,np.arange(N));filtered=np.zeros(N,bool)
for _,a,b in BLOCKS:filtered[a:b+1]=denv['filter_segment'](probability[a:b+1]>=.5,25)
reference_path=ROOT/'local_review/experiments/016_dead_broadcast_features/public_p_D.npy'
if reference_path.exists():
 reference=np.load(reference_path);assert np.allclose(probability,reference,atol=1e-6)
np.save(OUT/'p_D.npy',probability);np.save(OUT/'dead_mask.npy',filtered)
del denv,dmodel
# Keep the previously chosen H/A strategy frozen; no reselection.
# Сохраняем ранее выбранную стратегию H/A без нового подбора.
gt=pm.read_label_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')


In [ ]:
# Run preparation and algorithm definitions only, keeping all outputs in 019.
# Запускаем только подготовку и определения 007; все выходы остаются в 019.
progress('007 preparation')
parent007=json.loads((ROOT/'sol/007_clean_anchors/experiment.ipynb').read_text())
for index in [1,3,4,6,8]:
 source=''.join(parent007['cells'][index]['source'])
 if index==1:
  source=source.replace('EXPERIMENT = "007_clean_anchors"','EXPERIMENT = "020_penalty_gated_corridor"')
  source=source.replace('ROOT / "sol" / EXPERIMENT / "grid.json"','ROOT / "sol/007_clean_anchors/grid.json"')
  source=source.replace('import matplotlib.pyplot as plt\n','')
 exec(compile(source,f'007 cell {index}','exec'))
# Gate the direct corridor assignment by the CHOSEN team's own defensive line.
# Прямое правило коридора разрешено только по линии ВЫБРАННОЙ команды.
# The line must be strictly beyond its own penalty-area depth, toward midfield.
# Линия должна быть строго дальше границы своей штрафной в сторону центра поля.
PENALTY_DEPTH_M=16.5
BOUNDARY_TOLERANCE_M=1e-9
def line_is_advanced(line_x,defends_left,length):
 depth=np.asarray(line_x)*length if defends_left else (1-np.asarray(line_x))*length
 return np.isfinite(depth)&(depth>PENALTY_DEPTH_M+BOUNDARY_TOLERANCE_M)
# Mirror-invariant checks: inside, exactly on boundary, beyond, and missing.
# Проверки симметрии сторон: внутри, на границе, снаружи и пропуск.
probe=np.array([10.,16.5,16.51,30.,np.nan])/LENGTH
for left in [True,False]:
 test=probe if left else 1-probe
 assert np.array_equal(line_is_advanced(test,left,LENGTH),[False,False,True,True,False])
original_outside_owner=outside_owner.copy()
line_eligible=np.zeros((N,2),bool)
for half,first,last in BLOCKS:
 for team in [0,1]:
  line_eligible[first:last+1,team]=line_is_advanced(last_outfield[first:last+1,team],sides[half][team]=='left',LENGTH)
had_direct=outside_owner!=3
rows=np.flatnonzero(had_direct)
allowed=np.zeros(N,bool);allowed[rows]=line_eligible[rows,outside_owner[rows]]
# Removing forced evidence returns the frame to ordinary confident/contested processing.
# Убираем принудительное свидетельство: кадр возвращается к обычным уверенным/спорным.
outside_owner[had_direct&~allowed]=3
assert np.all(outside_owner[allowed]==original_outside_owner[allowed])
assert np.all(outside_owner[had_direct&~allowed]==3)
np.save(OUT/'corridor_eligible.npy',allowed)
np.save(OUT/'corridor_suppressed.npy',had_direct&~allowed)

# Replace legacy D completely before running the H/A solver.
# Полностью заменяем старый D перед запуском алгоритма H/A.
dead=filtered.copy()
selected=json.loads((ROOT/'sol/007_clean_anchors/selected_training_config.json').read_text())
progress('020 full inference')
labels,stats,gaps=predict_interval(selected,0,N,collect=True)
original_x=labels=='X';fallback_counts={};graded=np.zeros(N,bool)
def fill_nearest_time(values):
 valid=np.flatnonzero(np.isin(values,['H','A']))
 assert len(valid)>0
 missing=np.flatnonzero(~np.isin(values,['H','A']))
 if len(missing):
  right=np.searchsorted(valid,missing);lo=valid[np.maximum(right-1,0)];hi=valid[np.minimum(right,len(valid)-1)]
  chosen=np.where(missing-lo<=hi-missing,lo,hi)
  values[missing]=values[chosen]
 return values
# Fill only missing labels. Never change a confident anchor or a D decision.
# Заполняем только отсутствующие метки: уверенные якоря и решения D неизменны.
for half,first,last in BLOCKS:
 graded[first:last+1]=True
 nearest_evidence=np.full(last-first+1,'X',dtype='<U1')
 d=nearest[first:last+1];available=np.isfinite(d).any(axis=1)
 winner=np.argmin(d,axis=1)
 nearest_evidence[available]=np.array(['H','A'])[winner[available]]
 forced=outside_owner[first:last+1]!=3
 nearest_evidence[forced]=np.array(['H','A'])[outside_owner[first:last+1][forced]]
 assert np.isin(nearest_evidence,['H','A']).any(), 'No team evidence in entire half'
 half_prior=fill_nearest_time(nearest_evidence.copy())
 counts=dict(anchor_edge_fill=0,nearest_ball_fallback=0,same_half_fallback=0)
 states=dead[first:last+1]
 for a,b in runs_of(states):
  if states[a]:continue
  lo,hi=first+a,first+b;part=labels[lo:hi].copy();missing=part=='X'
  if not missing.any():continue
  if np.isin(part,['H','A']).any():
   counts['anchor_edge_fill']+=int(missing.sum());part=fill_nearest_time(part)
  else:
   evidence=nearest_evidence[a:b].copy()
   if np.isin(evidence,['H','A']).any():
    counts['nearest_ball_fallback']+=len(part);part=fill_nearest_time(evidence)
   else:
    counts['same_half_fallback']+=len(part);part=half_prior[a:b].copy()
  labels[lo:hi]=part
 fallback_counts[str(half)]=counts
labels[~graded]='D'
assert not np.any(labels=='X')
assert np.array_equal(labels[graded]=='D',filtered[graded])
assert np.array_equal(labels[graded&~original_x],predict_interval(selected,0,N)[0][graded&~original_x])
result=pd.DataFrame({'frame':np.arange(N),'label':labels});assert pm.validate_submission(result,N) is None
result.to_csv(OUT/'submission.csv',index=False)
# Exact start-of-half audit, including the second half pre-kickoff coverage.
# Точная проверка начал таймов, включая участок до второго розыгрыша.
audit={str(h):dict(first_frame=a,last_frame=b,first_label=str(labels[a]),X_frames=int((labels[a:b+1]=='X').sum()),initial_10s_X=int((labels[a:a+250]=='X').sum())) for h,a,b in BLOCKS}
(OUT/'coverage.json').write_text(json.dumps(dict(total_X=int((labels=='X').sum()),original_X_in_halves=int((original_x&graded).sum()),fallback_counts=fallback_counts,halves=audit,outside_half_policy='D'),indent=2))
print(json.dumps(audit,indent=2))


In [ ]:
# Metrics use only public labels, with full offline inference context.
# Метрики считаются только по публичным меткам с полным офлайн-контекстом.
def score_frames(frames):
 y=pm.label_codes(gt.set_index('frame').loc[frames,'label'].to_numpy())
 p=pm.label_codes(result.set_index('frame').loc[frames,'label'].to_numpy())
 runs=pm.valid_runs(y);cm=pm.scored_confusion(pm.confusion(y,p))
 out=dict(spellF1=pm.spell_scores(y,pm.fill_abstentions(p,runs),runs)['mean_f1'],macroF1=pm.macro_f1(cm),classes={})
 for i,label in enumerate(['H','A','D']):
  precision,recall,f1=pm.per_class_prf(cm,i)
  out['classes'][label]=dict(precision=precision,recall=recall,F1=f1,support=int(cm[i].sum()),predicted=int(cm[:,i].sum()))
 assert np.isclose(np.mean([v['F1'] for v in out['classes'].values()]),out['macroF1'])
 return out
public=gt.loc[gt.label.isin(['H','A','D']),'frame'].to_numpy()
metrics=dict(public_5min=score_frames(public),last_minute=score_frames(public[6000:]),
 context='016 trained on public prefix; H/A selected on public prefix; last minute repeatedly viewed diagnostic; full offline inputs, no refit')
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2))
config=dict(D_parent='016_dead_broadcast_features',D_model='public_model.pkl',threshold=.5,min_dead_frames=25,max_live_gap_frames=10,
 HA_parent='007_clean_anchors',HA_config=selected,corridor_min_own_goal_depth_m=16.5,corridor_comparison='strictly_greater',corridor_boundary_tolerance_m=1e-9,no_X_policy='interval anchor extension, nearest-ball fallback, same-half temporal fallback; D outside halves',
 model_sha256=hashlib.sha256((ROOT/'sol/016_dead_broadcast_features/public_model.pkl').read_bytes()).hexdigest())
(OUT/'model_config.json').write_text(json.dumps(config,indent=2))
progress('completed',X_frames=0)
print(json.dumps(metrics,indent=2))

# Compare to the unchanged 019 output; D and the no-X contract must stay exact.
# Сравниваем с неизменённым 019: D и отсутствие X должны сохраниться точно.
prior=pd.read_csv(ROOT/'local_review/experiments/019_boost016_clean007/submission.csv').label.to_numpy()
assert np.array_equal(prior=='D',labels=='D')
assert not (labels=='X').any()
change_summary={}
for name,mask in [('full_match',np.ones(N,bool)),('public_5min',np.isin(np.arange(N),public)),('last_minute',np.isin(np.arange(N),public[6000:]))]:
 change_summary[name]=dict(changed_labels=int(((prior!=labels)&mask).sum()),
  old_direct_live_frames=int((had_direct&~dead&mask).sum()),
  disabled_direct_live_frames=int((had_direct&~allowed&~dead&mask).sum()),
  remaining_direct_live_frames=int((allowed&~dead&mask).sum()))
(OUT/'change_summary.json').write_text(json.dumps(change_summary,indent=2))
seq=[]
for name,values in [('020',labels[public]),('019',prior[public]),('reference',gt.set_index('frame').loc[public,'label'].to_numpy())]:
 for label in ['H','A','D','X']:
  count=sum(values[a]==label for a,b in runs_of(values))
  seq.append(dict(model=name,label=label,sequences=int(count),frames=int((values==label).sum())))
pd.DataFrame(seq).to_csv(OUT/'sequence_counts_5min.csv',index=False)
print(json.dumps(change_summary,indent=2))
